In [2]:
"""
Minority Game con abstencion - TRES CASOS en un mismo codigo
+ funcion de informacion mutua
============================================================
Tesis: Complejidad e informacion en un minority game con opcion de abstencion
Base: R. Mansilla, Phys. Rev. E 62, 4553 (2000), Ec. (6)

Alfabeto: 0 = vender, 1 = comprar, 2 = abstenerse

Los tres casos:
  CASO 1. Agentes INGENUOS: sin estrategias ni puntaje. En cada ciclo cada
          agente elige 0, 1 o 2 al azar, con probabilidad 1/3.
  CASO 2. Agentes con 2 ESTRATEGIAS, SIN puntaje: los puntajes se quedan
          en 0, asi que en cada ciclo cada agente toma al azar una de sus
          2 estrategias y hace lo que dice para la historia actual.
  CASO 3. Agentes con 2 ESTRATEGIAS, CON puntaje: reglas del codigo base.
          Cada agente usa su estrategia con mas puntos, y en cada ciclo
          se suma un punto a TODAS las estrategias que habrian acertado.

Para cada caso:
  1. Corre 100 partidas, cada una con una semilla distinta (las mismas
     100 semillas en los tres casos).
  2. A cada cadena de ganadores le aplica M(d), Ec. (6) del PRE 62,
     para d = 1, ..., 100.
  3. Promedia M(d) sobre las 100 cadenas y calcula el intervalo de
     confianza del 95%.
  4. Guarda los resultados en CSV y hace dos graficas.

Comun a los tres casos (igual que el codigo base):
  - N = 50, m = 3, K = 3
  - la regla de ganador: argmin con desempate aleatorio
  - el desplazamiento de la memoria
  - se registra desde el primer ciclo (sin burn-in)
"""

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    from scipy.stats import t as t_student
    T_95 = None          # se calcula con scipy mas abajo
except ImportError:
    t_student = None
    T_95 = 1.984         # t de Student, 95%, 99 grados de libertad


# =====================================================================
# PARAMETROS COMUNES
# =====================================================================
SEMILLA_MAESTRA = 1907443528   # genera las 100 semillas de forma reproducible
N_CADENAS = 100                # numero de cadenas (semillas distintas)

N       = 50     # agentes
M       = 3      # memoria (brain size)
K       = 3      # tamano del alfabeto
BURN_IN = 0      # ciclos iniciales que se descartan (0 = desde el primero)

D_MAX   = 100    # M(d) se calcula para d = 1, ..., D_MAX
NIVEL   = 0.95   # nivel de confianza del intervalo


# =====================================================================
# PARAMETROS DE CADA CASO
# =====================================================================
# T y S de cada caso se dejaron tal como estaban en el notebook
CASOS = [
    {
        "nombre":  "ingenuos",
        "T":       6000,
        "S":       0,          # sin estrategias
        "puntaje": False,
        "encabezado": "Agentes ingenuos (eleccion al azar)",
        "titulo":  "agentes ingenuos",
        "csv_promedio": "Md_ingenuos_promedio.csv",
        "csv_todas":    "Md_ingenuos_todas_las_cadenas.csv",
        "png_promedio": "grafica_Md_promedio.png",
        "png_ic":       "grafica_Md_intervalo_confianza.png",
        "ylim_cero": False,
    },
    {
        "nombre":  "2 estrategias sin puntaje",
        "T":       6000,
        "S":       2,
        "puntaje": False,
        "encabezado": "Agentes con 2 estrategias, SIN recompensa, sin burn-in",
        "titulo":  "2 estrategias, sin recompensa",
        "csv_promedio": "Md_estrategias_sin_recompensa_promedio.csv",
        "csv_todas":    "Md_estrategias_sin_recompensa_todas_las_cadenas.csv",
        "png_promedio": "grafica_Md_estrategias_sin_recompensa_promedio.png",
        "png_ic":       "grafica_Md_estrategias_sin_recompensa_intervalo_confianza.png",
        "ylim_cero": True,
    },
    {
        "nombre":  "2 estrategias con puntaje",
        "T":       6000,
        "S":       2,
        "puntaje": True,
        "encabezado": "Agentes con 2 estrategias, CON puntaje, sin burn-in",
        "titulo":  "2 estrategias, con puntaje",
        "csv_promedio": "Md_2estrategias_con_puntaje_promedio.csv",
        "csv_todas":    "Md_2estrategias_con_puntaje_todas_las_cadenas.csv",
        "png_promedio": "grafica_Md_2estrategias_con_puntaje_promedio.png",
        "png_ic":       "grafica_Md_2estrategias_con_puntaje_intervalo_confianza.png",
        "ylim_cero": True,
    },
]


# =====================================================================
# REGLA DE GANADOR  (identica a la del codigo base)
# =====================================================================
def regla_ganador(conteos, rng):
    """Gana el argmin sobre las K acciones, con desempate aleatorio."""
    empatados = np.flatnonzero(conteos == conteos.min())
    return int(rng.choice(empatados))


# =====================================================================
# CASO 1: AGENTES INGENUOS
# =====================================================================
def simular_ingenuos(N, m, K, T, burn_in, semilla):
    """Corre una partida con agentes que eligen al azar.

    Cada agente, en cada ciclo, elige 0, 1 o 2 con probabilidad 1/3.
    No hay estrategias, no hay puntajes, no hay aprendizaje.

    La memoria (historia de los ultimos m ganadores) se sigue llevando
    igual que en el codigo base, para que la estructura sea la misma y
    despues solo haya que cambiar la forma de decidir. Pero aqui nadie
    la consulta.
    """
    rng = np.random.default_rng(semilla)

    historia = rng.integers(0, K, size=m)
    serie_ganadores = np.zeros(T, dtype=int)

    for t in range(burn_in + T):

        # 1. cada agente elige al azar entre 0, 1 y 2
        acciones = rng.integers(0, K, size=N)

        # 2. se cuenta cuantos eligieron cada opcion
        conteos = np.bincount(acciones, minlength=K)

        # 3. gana la minoria (misma regla que el codigo base)
        ganador = regla_ganador(conteos, rng)

        # 4. registrar solo despues del transitorio
        if t >= burn_in:
            serie_ganadores[t - burn_in] = ganador

        # 5. la memoria se desplaza: entra el ganador, sale el mas viejo
        historia = np.roll(historia, -1)
        historia[-1] = ganador

    return serie_ganadores


# =====================================================================
# CASO 2: s ESTRATEGIAS POR AGENTE, SIN PUNTAJE
# =====================================================================
def simular_sin_recompensa(N, m, s, K, T, semilla):
    """Corre una partida con agentes que tienen s estrategias pero no
    las califican.

    Todo es igual al codigo base, excepto que se quita la recompensa:
    los puntajes valen 0 siempre. Entonces en el paso 2 las s estrategias
    de cada agente estan empatadas y el ruido decide cual se usa. Es decir,
    en cada ciclo cada agente toma al azar una de sus estrategias y hace
    lo que esa estrategia dice para la historia actual.

    Diferencia clave con los agentes ingenuos: aqui la decision SI depende
    de la memoria, porque las estrategias son tablas que responden a la
    historia de los ultimos m ganadores.
    """
    rng = np.random.default_rng(semilla)
    N_HIST = K ** m

    # --- agentes (igual que el codigo base) ---
    # estrategias[i, j, h] = accion que la estrategia j del agente i
    #                        recomienda ante la historia con indice h
    estrategias = rng.integers(0, K, size=(N, s, N_HIST))

    # puntajes[i, j]: se quedan en 0 porque no hay recompensa
    puntajes = np.zeros((N, s))

    # --- memoria: los m datos que el mercado tiene "al nacer" ---
    historia = rng.integers(0, K, size=m)

    pesos = K ** np.arange(m - 1, -1, -1)   # convierte historia -> indice
    indices = np.arange(N)

    serie_ganadores = np.zeros(T, dtype=int)

    for t in range(T):

        # 1. que situacion estan viendo los agentes
        idx = int(historia @ pesos)

        # 2. cada agente toma su estrategia con mas puntos.
        #    Como todos los puntajes valen 0, el ruido decide al azar.
        mejor = np.argmax(puntajes + rng.random((N, s)) * 0.5, axis=1)

        # 3. consulta esa estrategia en la historia actual y actua
        acciones = estrategias[indices, mejor, idx]

        # 4. se cuenta cuantos eligieron cada opcion
        conteos = np.bincount(acciones, minlength=K)

        # 5. gana la minoria
        ganador = regla_ganador(conteos, rng)

        # 6. (SIN RECOMPENSA: en el codigo base aqui se sumaban puntos)

        # 7. registrar desde el primer ciclo
        serie_ganadores[t] = ganador

        # 8. la memoria se desplaza: entra el ganador, sale el mas viejo
        historia = np.roll(historia, -1)
        historia[-1] = ganador

    return serie_ganadores


# =====================================================================
# CASO 3: s ESTRATEGIAS POR AGENTE, CON PUNTAJE
# =====================================================================
def simular_con_puntaje(N, m, s, K, T, semilla):
    """Corre una partida con las reglas del codigo base, sin burn-in.

    Cada agente tiene s estrategias. En cada ciclo usa la que tiene mas
    puntos (los empates se rompen al azar), y al final del ciclo todas
    las estrategias que habrian acertado al ganador reciben un punto.

    Devuelve la serie de ganadores y cuantas veces acerto cada agente.
    """
    rng = np.random.default_rng(semilla)
    N_HIST = K ** m

    # --- agentes (igual que el codigo base) ---
    # estrategias[i, j, h] = accion que la estrategia j del agente i
    #                        recomienda ante la historia con indice h
    estrategias = rng.integers(0, K, size=(N, s, N_HIST))

    # puntajes[i, j] = puntos virtuales (casos de exito) de esa estrategia
    puntajes = np.zeros((N, s))

    # --- memoria: los m datos que el mercado tiene "al nacer" ---
    historia = rng.integers(0, K, size=m)

    pesos = K ** np.arange(m - 1, -1, -1)   # convierte historia -> indice
    indices = np.arange(N)

    serie_ganadores = np.zeros(T, dtype=int)
    exitos_agente = np.zeros(N, dtype=int)

    for t in range(T):

        # 1. que situacion estan viendo los agentes
        idx = int(historia @ pesos)

        # 2. cada agente toma su estrategia con mas puntos.
        #    El ruido < 1 rompe empates al azar sin alterar el orden
        #    entre puntajes enteros distintos.
        mejor = np.argmax(puntajes + rng.random((N, s)) * 0.5, axis=1)

        # 3. consulta esa estrategia en la historia actual y actua
        acciones = estrategias[indices, mejor, idx]

        # 4. se cuenta cuantos eligieron cada opcion
        conteos = np.bincount(acciones, minlength=K)

        # 5. gana la minoria
        ganador = regla_ganador(conteos, rng)

        # 6. se premian TODAS las estrategias que hubieran acertado,
        #    se hayan jugado o no (regla del codigo base / PRE 62).
        puntajes += (estrategias[:, :, idx] == ganador)

        # 7. registrar desde el primer ciclo
        serie_ganadores[t] = ganador
        exitos_agente += (acciones == ganador)

        # 8. la memoria se desplaza: entra el ganador, sale el mas viejo
        historia = np.roll(historia, -1)
        historia[-1] = ganador

    return serie_ganadores, exitos_agente


# =====================================================================
# FUNCION DE INFORMACION MUTUA  -  Ec. (6) del PRE 62
# =====================================================================
def informacion_mutua(serie, d, K):
    """M(d) = sum_{a,b} P_ab(d) * log2[ P_ab(d) / (P_a * P_b) ]

    P_ab(d): probabilidad de encontrar el simbolo a y, d lugares despues,
             el simbolo b. Se estima contando todos los pares
             (serie[i], serie[i+d]) de la cadena.
    P_a:     densidad del simbolo a en la cadena completa (como dice
             Mansilla: "the density of the symbol alpha").

    La suma corre sobre los K*K = 9 pares (a, b). Los pares que nunca
    aparecen (P_ab = 0) no aportan, porque 0 * log(0) = 0.
    """
    x = serie[:-d]          # simbolo alpha
    y = serie[d:]           # simbolo beta, d lugares despues
    n_pares = len(x)

    # probabilidades conjuntas P_ab(d): matriz K x K
    conjunta = np.bincount(x * K + y, minlength=K * K).reshape(K, K) / n_pares

    # densidades P_a de cada simbolo en toda la cadena
    densidad = np.bincount(serie, minlength=K) / len(serie)
    producto = np.outer(densidad, densidad)      # P_a * P_b

    M_d = 0.0
    for a in range(K):
        for b in range(K):
            if conjunta[a, b] > 0:
                M_d += conjunta[a, b] * np.log2(conjunta[a, b] / producto[a, b])
    return M_d


def sesgo_esperado(K, n_pares):
    """Valor que da M(d) en una cadena SIN correlacion, solo por ser finita.

    Aunque la cadena sea totalmente aleatoria, M(d) no sale exactamente 0:
    con un numero finito de pares siempre hay fluctuaciones. Su valor
    esperado es aproximadamente (K-1)^2 / (2 * n * ln 2).
    Sirve como linea de referencia en la grafica.
    """
    return (K - 1) ** 2 / (2 * n_pares * np.log(2))


# =====================================================================
# UNA PARTIDA SEGUN EL CASO
# =====================================================================
def simular_caso(caso, semilla):
    """Llama a la simulacion que corresponde al caso.

    Devuelve (serie_ganadores, exitos_agente). En los casos sin puntaje
    exitos_agente es None.
    """
    T, S = caso["T"], caso["S"]
    if S == 0:
        return simular_ingenuos(N, M, K, T, BURN_IN, semilla), None
    if not caso["puntaje"]:
        return simular_sin_recompensa(N, M, S, K, T, semilla), None
    return simular_con_puntaje(N, M, S, K, T, semilla)


# =====================================================================
# ANALISIS COMPLETO DE UN CASO
# =====================================================================
def analizar_caso(caso, semillas, distancias, t_crit):
    T, S = caso["T"], caso["S"]
    texto_s = f"  s={S}" if S else ""

    print("\n" + "=" * 62)
    print(f"  {caso['encabezado']}")
    print(f"  N={N}  m={M}{texto_s}  K={K}  T={T}  burn_in={BURN_IN}")
    print(f"  cadenas={N_CADENAS}  d = 1..{D_MAX}")
    print("=" * 62)

    # tabla[c, j] = M(d = j+1) de la cadena c
    tabla = np.zeros((N_CADENAS, D_MAX))
    frecuencias = np.zeros((N_CADENAS, K))
    tasa_exito = np.zeros(N_CADENAS)

    for c, semilla in enumerate(semillas):
        serie, exitos = simular_caso(caso, int(semilla))
        frecuencias[c] = np.bincount(serie, minlength=K) / T
        if exitos is not None:
            tasa_exito[c] = exitos.sum() / (N * T)
        for j, d in enumerate(distancias):
            tabla[c, j] = informacion_mutua(serie, d, K)
        if (c + 1) % 10 == 0:
            print(f"  cadenas terminadas: {c + 1}/{N_CADENAS}")

    # --- promedio sobre las 100 cadenas, para cada d ---
    media = tabla.mean(axis=0)
    desv = tabla.std(axis=0, ddof=1)          # desviacion estandar muestral
    error_est = desv / np.sqrt(N_CADENAS)     # error estandar de la media

    # --- intervalo de confianza (t de Student) ---
    ic_inf = media - t_crit * error_est
    ic_sup = media + t_crit * error_est

    referencia = sesgo_esperado(K, T - distancias)

    # --- resumen en pantalla ---
    print("\n  Frecuencia media de cada simbolo (100 cadenas):")
    for a, nombre in enumerate(["vender", "comprar", "abstenerse"]):
        print(f"    {a} ({nombre:10s}): {frecuencias[:, a].mean():.4f}")
    if caso["puntaje"]:
        print(f"\n  Tasa de exito media de los agentes: {tasa_exito.mean():.4f}"
              f"  (agentes al azar: 0.265)")
    print(f"\n  M(d) promedio: min={media.min():.3e}  max={media.max():.3e}"
          f"  media global={media.mean():.3e} bits")
    print(f"  M(1)={media[0]:.3e}  M(2)={media[1]:.3e}  M(3)={media[2]:.3e}"
          f"  M(4)={media[3]:.3e}")
    print(f"  Sesgo esperado de una cadena sin correlacion: "
          f"{referencia.mean():.3e} bits")
    print(f"  Techo teorico de M(d) con K=3: log2(3) = {np.log2(K):.4f} bits")

    # --- guardar resultados ---
    parametros = f"N={N} m={M}{' s=' + str(S) if S else ''} K={K} T={T} burn_in={BURN_IN}"
    encabezado = (
        f"M(d) con {caso['nombre']}, Ec. (6) PRE 62\n"
        f"{parametros} cadenas={N_CADENAS} semilla_maestra={SEMILLA_MAESTRA}\n"
        f"intervalo de confianza {int(NIVEL*100)}% con t de Student "
        f"(t={t_crit:.4f})\n"
        f"d,media,desv_estandar,error_estandar,ic_inferior,ic_superior,"
        f"sesgo_esperado"
    )
    np.savetxt(caso["csv_promedio"],
               np.column_stack([distancias, media, desv, error_est,
                                ic_inf, ic_sup, referencia]),
               fmt=["%d"] + ["%.8e"] * 6, delimiter=",",
               header=encabezado, comments="# ")

    encabezado_todas = (
        f"M(d) de cada cadena. Fila = una semilla, columna = d\n"
        f"{parametros} {caso['nombre']}\n"
        f"semilla," + ",".join(f"d{d}" for d in distancias)
    )
    np.savetxt(caso["csv_todas"],
               np.column_stack([semillas.astype(np.int64), tabla]),
               fmt=["%d"] + ["%.8e"] * D_MAX, delimiter=",",
               header=encabezado_todas, comments="# ")

    print(f"\n  Guardado: {caso['csv_promedio']}")
    print(f"  Guardado: {caso['csv_todas']}")

    # =================================================================
    # GRAFICAS
    # =================================================================
    AZUL = "#2a78d6"
    GRIS = "#52514e"
    TINTA = "#0b0b0b"

    plt.rcParams.update({
        "font.size": 11,
        "axes.edgecolor": GRIS,
        "axes.labelcolor": TINTA,
        "xtick.color": GRIS,
        "ytick.color": GRIS,
        "axes.spines.top": False,
        "axes.spines.right": False,
    })

    if S:
        subtitulo = (f"N={N}, m={M}, s={S}, K={K}, T={T}, sin burn-in, "
                     f"{N_CADENAS} semillas")
    else:
        subtitulo = f"N={N}, m={M}, K={K}, T={T}, {N_CADENAS} semillas"

    # --- Grafica 1: promedio de M(d) ---
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(distancias, media, color=AZUL, lw=2, marker="o", ms=4,
            label="Promedio de 100 cadenas")
    ax.plot(distancias, referencia, color=GRIS, lw=1.5, ls="--",
            label="Sesgo esperado de una cadena sin correlación")
    ax.set_xlabel("Distancia d")
    ax.set_ylabel("M(d)  [bits]")
    ax.set_title(f"Información mutua promedio — {caso['titulo']}\n"
                 + subtitulo, loc="left", color=TINTA)
    ax.set_xlim(0, D_MAX + 1)
    if caso["ylim_cero"]:
        ax.set_ylim(bottom=0)
    ax.grid(axis="y", color="#e5e4df", lw=0.8)
    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.14),
              ncol=3, fontsize=9)
    fig.tight_layout()
    fig.savefig(caso["png_promedio"], dpi=200)
    plt.close(fig)

    # --- Grafica 2: promedio con intervalo de confianza ---
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.fill_between(distancias, ic_inf, ic_sup, color=AZUL, alpha=0.18,
                    lw=0, label=f"Intervalo de confianza {int(NIVEL*100)}%")
    ax.errorbar(distancias, media, yerr=t_crit * error_est, fmt="o",
                ms=4, color=AZUL, ecolor=AZUL, elinewidth=1, capsize=2,
                label="Promedio de 100 cadenas")
    ax.plot(distancias, referencia, color=GRIS, lw=1.5, ls="--",
            label="Sesgo esperado de una cadena sin correlación")
    ax.set_xlabel("Distancia d")
    ax.set_ylabel("M(d)  [bits]")
    ax.set_title(f"Promedio de M(d) con intervalo de confianza del "
                 f"{int(NIVEL*100)}% — {caso['titulo']}\n"
                 f"t de Student con {N_CADENAS - 1} grados de libertad",
                 loc="left", color=TINTA)
    ax.set_xlim(0, D_MAX + 1)
    if caso["ylim_cero"]:
        ax.set_ylim(bottom=0)
    ax.grid(axis="y", color="#e5e4df", lw=0.8)
    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.14),
              ncol=3, fontsize=9)
    fig.tight_layout()
    fig.savefig(caso["png_ic"], dpi=200)
    plt.close(fig)

    print(f"  Guardado: {caso['png_promedio']}")
    print(f"  Guardado: {caso['png_ic']}")

    return media


# =====================================================================
# PROGRAMA PRINCIPAL
# =====================================================================
if __name__ == "__main__":

    # --- las 100 semillas, generadas de forma reproducible ---
    #     (las mismas en los tres casos)
    semillas = np.random.SeedSequence(SEMILLA_MAESTRA).generate_state(N_CADENAS)
    distancias = np.arange(1, D_MAX + 1)

    # --- valor critico de la t de Student ---
    t_crit = (t_student.ppf(0.5 + NIVEL / 2, N_CADENAS - 1)
              if t_student is not None else T_95)

    for caso in CASOS:
        analizar_caso(caso, semillas, distancias, t_crit)



  Agentes ingenuos (eleccion al azar)
  N=50  m=3  K=3  T=6000  burn_in=0
  cadenas=100  d = 1..100
  cadenas terminadas: 10/100
  cadenas terminadas: 20/100
  cadenas terminadas: 30/100
  cadenas terminadas: 40/100
  cadenas terminadas: 50/100
  cadenas terminadas: 60/100
  cadenas terminadas: 70/100
  cadenas terminadas: 80/100
  cadenas terminadas: 90/100
  cadenas terminadas: 100/100

  Frecuencia media de cada simbolo (100 cadenas):
    0 (vender    ): 0.3336
    1 (comprar   ): 0.3341
    2 (abstenerse): 0.3323

  M(d) promedio: min=4.047e-04  max=5.629e-04  media global=4.865e-04 bits
  M(1)=4.839e-04  M(2)=4.818e-04  M(3)=4.529e-04  M(4)=4.858e-04
  Sesgo esperado de una cadena sin correlacion: 4.850e-04 bits
  Techo teorico de M(d) con K=3: log2(3) = 1.5850 bits

  Guardado: Md_ingenuos_promedio.csv
  Guardado: Md_ingenuos_todas_las_cadenas.csv
  Guardado: grafica_Md_promedio.png
  Guardado: grafica_Md_intervalo_confianza.png

  Agentes con 2 estrategias, SIN recompensa, sin 